# 04_holdout_suzuka_2024

Evaluate the model on Suzuka 2024 held out from training.

In [17]:
import pandas as pd
import joblib
from pathlib import Path

FEATURES_PATH = Path("../data/processed/driver_race_features.csv")
HOLDOUT_PATH = Path("../data/processed/holdout_suzuka_2024.csv")
MODEL_PATH = Path("../data/processed/final_model_logreg.joblib")

if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Holdout file missing: {HOLDOUT_PATH}. Re-run notebook 03 after creating the holdout.")

features = pd.read_csv(FEATURES_PATH)
holdout_df = pd.read_csv(HOLDOUT_PATH)
model = joblib.load(MODEL_PATH)

print("Features shape:", features.shape)
print("Holdout shape:", holdout_df.shape)
print("Model:", type(model))

Features shape: (25121, 17)
Holdout shape: (20, 17)
Model: <class 'sklearn.pipeline.Pipeline'>


In [18]:
# Define columns and holdout mask to match training drop
HOLDOUT_SEASON = 2024
HOLDOUT_CIRCUIT = "suzuka"

mask = (features["season"] == HOLDOUT_SEASON) & (features["circuitId"] == HOLDOUT_CIRCUIT)

drop_cols = [
    "target_top10",
    "race_date",
    "driver_name",
    "constructor_name",
    "circuit_name",
    "raceId",
]
cat_cols = ["circuitId", "era", "grid_bucket"]

In [19]:
# Build training feature space (same as training notebook, without holdout)
train_X = features.loc[~mask].drop(columns=drop_cols, errors="ignore")
train_X = pd.get_dummies(train_X, columns=cat_cols, drop_first=True)
train_cols = train_X.columns
print("Train feature space:", train_X.shape)

Train feature space: (25121, 91)


In [20]:
# Prepare holdout features and align columns
X_new = holdout_df.drop(columns=drop_cols, errors="ignore")
X_new = pd.get_dummies(X_new, columns=cat_cols, drop_first=True)

for col in train_cols:
    if col not in X_new:
        X_new[col] = 0
X_new = X_new[train_cols]

print("Holdout matrix:", X_new.shape)

Holdout matrix: (20, 91)


In [21]:
# Predict probabilities and compare with actual target
proba = model.predict_proba(X_new)[:, 1]

preds = holdout_df[["driver_name", "constructor_name", "grid", "target_top10"]].copy()
preds["p_top10"] = proba
preds = preds.sort_values("p_top10", ascending=False).reset_index(drop=True)

preds

,driver_name,constructor_name,grid,target_top10,p_top10
0,Sergio Pérez,Red Bull,2,1,0.864593
1,Max Verstappen,Red Bull,1,1,0.842854
2,Fernando Alonso,Aston Martin,5,1,0.826445
3,Lando Norris,McLaren,3,1,0.802907
4,Carlos Sainz,Ferrari,4,1,0.787496
5,Lewis Hamilton,Mercedes,7,1,0.761566
6,Charles Leclerc,Ferrari,8,1,0.697790
7,Oscar Piastri,McLaren,6,1,0.668748
8,Valtteri Bottas,Sauber,13,0,0.578777
9,George Russell,Mercedes,9,1,0.571748


Notes:
- Re-run notebook 03 after the holdout change to retrain and save the model without Suzuka 2024.
- This notebook assumes `driver_race_features.csv` contains Suzuka 2024 rows and the model file was produced after excluding them from training.